# Card Distribution Analysis

This notebook analyzes the distribution of turned cards to check if all cards appear uniformly.

In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from scipy import stats
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Load data
stats_dir = project_root / "stats"
analysis_files = list(stats_dir.glob("analysis_*.json")) if stats_dir.exists() else []

if analysis_files:
    latest_file = max(analysis_files, key=lambda p: p.stat().st_mtime)
    with open(latest_file, 'r') as f:
        data = json.load(f)
    print(f"Loaded: {latest_file.name}")
else:
    print("No data files found")
    data = None

## Card Distribution Statistics

In [ ]:
if data:
    card_dist = data.get('card_distribution_analysis', {})
    
    if 'error' not in card_dist:
        print("=" * 60)
        print("CARD DISTRIBUTION ANALYSIS")
        print("=" * 60)
        
        print(f"\nTotal Cards Turned: {card_dist.get('total_turned', 'N/A')}")
        print(f"Expected per card: {card_dist.get('expected_per_card', 'N/A'):.2f}")
        print(f"Uniform Distribution: {card_dist.get('is_uniform', 'N/A')}")
        print(f"Chi-square p-value: {card_dist.get('chi_square_p_value', 'N/A'):.4f}")
        
        print("\nMost Common Cards:")
        for card, count in card_dist.get('most_common', [])[:10]:
            print(f"  {card}: {count} times")
        
        print("\nLeast Common Cards:")
        for card, count in card_dist.get('least_common', [])[-10:]:
            print(f"  {card}: {count} times")
        
        never_appeared = card_dist.get('never_appeared', [])
        if never_appeared:
            print(f"\nNever Appeared ({len(never_appeared)} cards):")
            for card in never_appeared:
                print(f"  {card}")
        else:
            print("\n✓ All cards appeared at least once")

## Visualize Card Distribution

In [ ]:
if data and 'card_distribution_analysis' in data:
    card_dist = data['card_distribution_analysis']
    
    if 'card_counts' in card_dist:
        card_counts = card_dist['card_counts']
        expected = card_dist.get('expected_per_card', 0)
        
        # Create DataFrame
        df_cards = pd.DataFrame([
            {'card': card, 'count': count, 'expected': expected}
            for card, count in card_counts.items()
        ])
        df_cards = df_cards.sort_values('count')
        
        # Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        # Bar chart
        ax1.bar(range(len(df_cards)), df_cards['count'], alpha=0.7, label='Observed')
        ax1.axhline(y=expected, color='r', linestyle='--', label=f'Expected ({expected:.1f})')
        ax1.set_xlabel('Card (sorted by frequency)', fontsize=12)
        ax1.set_ylabel('Count', fontsize=12)
        ax1.set_title('Card Distribution', fontsize=14, fontweight='bold')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Histogram
        ax2.hist(df_cards['count'], bins=20, alpha=0.7, edgecolor='black')
        ax2.axvline(x=expected, color='r', linestyle='--', label=f'Expected ({expected:.1f})')
        ax2.set_xlabel('Count', fontsize=12)
        ax2.set_ylabel('Number of Cards', fontsize=12)
        ax2.set_title('Distribution of Card Frequencies', fontsize=14, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nStandard deviation: {df_cards['count'].std():.2f}")
        print(f"Coefficient of variation: {(df_cards['count'].std() / df_cards['count'].mean() * 100):.2f}%")